# 01 — Data Exploration

## What we're doing here
Before writing any detection algorithm, we need to **see the data**.  
A candle-by-candle dataset is just a table — but charts reveal patterns the table hides.

This notebook is intentionally minimal. We just:
1. Load the OHLC dataset
2. Load a **labeled** version that tells us which candles belong to which scenario
3. Render a candlestick chart so the shapes become real

## The dataset
Each row is one candle (one bar / one period) with **OHLCV** fields:

| Column   | Meaning                              |
|----------|--------------------------------------|
| `open`   | Opening price of the period          |
| `high`   | Highest price reached in the period  |
| `low`    | Lowest price reached in the period   |
| `close`  | Closing price of the period          |
| `volume` | Number of units traded               |

## Why hand-crafted fixtures?
Real market data is messy — bugs hide easily.  
This series uses a small **hand-crafted dataset** where every scenario has a **known expected outcome**.  
If the algorithm flags something different than what's labeled, we know exactly what broke and why.

The scenarios cover:
- Classic shapes (textbook RBR, DBD, DBR, RBD)
- Edge cases (weak departure, doji bases, wide bases, nested bases)
- Negative cases (things that should NOT be flagged as zones)


## Setup
Install the libraries we'll use across the whole series:
- **pandas** — table loading and manipulation
- **numpy** — array math
- **plotly** — interactive charts
- **nbformat** — required by plotly for notebook rendering

Run this cell once per environment. The `%pip` magic installs into the active kernel.


In [5]:
%pip install pandas numpy plotly nbformat yfinance joblib tqdm

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (18 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-macosx_11_0_arm64.whl (2.6 MB)
Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl (181 kB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
U

## Load the raw OHLC fixtures
`fixtures.csv` is the bare price data — no labels, no indicators, just open/high/low/close per date.

We:
- Parse the first column as a **datetime index** so candles align on the time axis
- Print the shape to confirm how many rows we loaded
- Show the first 30 rows to eyeball the values


In [2]:
import pandas as pd

df = pd.read_csv("../fixtures.csv", index_col=0, parse_dates=True)
print(df.shape)  
df.head(10)


(98, 5)


,open,high,low,close,volume
Date,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000
2024-02-05 00:00:00+00:00,100.8,101.3,99.8,100.3,1000000
2024-02-12 00:00:00+00:00,100.3,101.4,99.8,100.9,1000000
2024-02-19 00:00:00+00:00,100.9,101.4,99.9,100.4,1000000
2024-02-26 00:00:00+00:00,100.4,101.5,99.9,101.0,1000000


## The labeled version
`fixtures_labeled.csv` is the **same OHLC rows**, plus two extra columns that tell us the ground truth:

| Column     | Meaning                                                              |
|------------|----------------------------------------------------------------------|
| `scenario` | Which hand-crafted scenario this candle belongs to (e.g. `A_demand_RBR`) |
| `note`     | What role this candle plays inside the scenario (`BASE`, `LEG-IN`, `LEG-OUT`, `warmup` …) |

These labels are **the test answers**. The algorithm itself only sees the raw OHLC — but we use the labels to check whether it found the right thing.


In [3]:

labeled = pd.read_csv("../fixtures_labeled.csv", index_col=0, parse_dates=True)
labeled[["scenario", "note"]].head(10)

,scenario,note
Date,,
2024-01-01 00:00:00+00:00,warmup,ATR warm-up
2024-01-08 00:00:00+00:00,warmup,ATR warm-up
2024-01-15 00:00:00+00:00,warmup,ATR warm-up
2024-01-22 00:00:00+00:00,warmup,ATR warm-up
2024-01-29 00:00:00+00:00,warmup,ATR warm-up
2024-02-05 00:00:00+00:00,warmup,ATR warm-up
2024-02-12 00:00:00+00:00,warmup,ATR warm-up
2024-02-19 00:00:00+00:00,warmup,ATR warm-up
2024-02-26 00:00:00+00:00,warmup,ATR warm-up


## Visualize the candles
Tables are useless for spotting price structure. We use **Plotly's candlestick chart** so each row becomes a real candle on a time axis.

Reading a candle:
- **Body** = rectangle between `open` and `close`
- **Green body** → `close > open` (bullish — price went up over the period)
- **Red body** → `close < open` (bearish — price went down)
- **Upper wick** (thin line above body) → reaches the period's `high`
- **Lower wick** (thin line below body) → reaches the period's `low`

What you should see scanning the chart:
- Long stretches of small, similar candles (warmups & bases — quiet periods)
- A few **strong impulse candles** (the legs into/out of zones)
- Occasional **tight clusters** where price pauses, then moves sharply (the actual S&D zones)

The rangeslider at the bottom is disabled to keep the view clean.


In [4]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC",
    )
])

fig.update_layout(
    title="Fixtures — Candlestick Chart",
    xaxis_title="Date",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    height=500,
)

fig.show()
